# CSBP441 - LN6: Gradients and Edge Detection

**Applied Computer Vision | UAE University**

[Open this notebook in Google Colab](https://colab.research.google.com/github/MoyoG/CSBP441/blob/main/docs/notebooks/LN6_Edge_Detection.ipynb)

This guided Colab notebook turns an uploaded image into edge maps while showing
what each detector computes. Run the cells from top to bottom, change the
parameters, and explain the observations in your own words.

## Learning outcomes

By the end of this notebook, you should be able to:

1. Explain an edge as a rapid spatial change in image intensity.
2. Compute horizontal and vertical image derivatives, `Gx` and `Gy`.
3. Calculate gradient magnitude and direction.
4. Compare Roberts, Prewitt, Sobel, and Scharr operators.
5. Explain Laplacian, LoG, DoG, and zero-crossing edge detection.
6. Explain the stages of the Canny edge detector.
7. Predict how smoothing, thresholds, and noise affect an edge map.
8. Convert an edge response into a binary edge mask and use it on an image.

> Important: an edge detector responds to **intensity changes**, not directly
> to semantic objects. Texture, shadows, highlights, and noise can therefore
> create edges too.



## 1. Setup

Colab already includes NumPy, Matplotlib, OpenCV, and ipywidgets. If a local
environment is missing a package, install it there before running the notebook.



In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from numpy.lib.stride_tricks import sliding_window_view

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.titlesize"] = 11

OUTPUT_DIR = Path("ln6_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)



## 2. Upload an image

In Google Colab, run the next cell and choose a `.jpg`, `.jpeg`, or `.png`
image. An image with objects, straight boundaries, curves, and some texture is
useful for comparison.

To keep the notebook responsive, very large images are resized so their longest
side is at most 640 pixels. Outside Colab, the cell creates a synthetic test
image so the notebook can still be checked.



In [ ]:
def make_demo_image(height=420, width=600):
    """Create a synthetic image only for running outside Google Colab."""
    image = np.full((height, width, 3), 225, dtype=np.uint8)
    cv2.rectangle(image, (45, 55), (245, 230), (35, 105, 210), -1)
    cv2.circle(image, (410, 150), 85, (220, 70, 45), -1)
    cv2.line(image, (40, 340), (560, 275), (20, 20, 20), 9)
    cv2.putText(image, "LN6", (330, 365), cv2.FONT_HERSHEY_SIMPLEX,
                2.1, (50, 145, 55), 6, cv2.LINE_AA)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def load_uploaded_rgb(max_side=640):
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab: using a synthetic test image.")
        image_rgb = make_demo_image()
        filename = "synthetic_ln6_image.png"
    else:
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No image was uploaded. Run the cell again and select an image.")
        filename, data = next(iter(uploaded.items()))
        encoded = np.frombuffer(data, dtype=np.uint8)
        image_bgr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise ValueError("OpenCV could not read this file. Please upload a JPG or PNG image.")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    height, width = image_rgb.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale < 1.0:
        image_rgb = cv2.resize(
            image_rgb,
            (round(width * scale), round(height * scale)),
            interpolation=cv2.INTER_AREA,
        )
    return filename, image_rgb


filename, image_rgb = load_uploaded_rgb()
gray_u8 = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
gray = gray_u8.astype(np.float32)

print("File:", filename)
print("RGB shape:", image_rgb.shape)
print("Grayscale shape:", gray.shape)
print("Intensity range:", float(gray.min()), "to", float(gray.max()))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Uploaded RGB image")
axes[1].imshow(gray_u8, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Grayscale intensity image")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()



## 3. What is an edge?

For a 1D intensity signal, the first derivative is large where intensity
changes quickly:

$$
\frac{dI}{dx} \approx I(x+1)-I(x)
$$

In a 2D image, intensity can change in both directions:

$$
G_x = \frac{\partial I}{\partial x}, \qquad
G_y = \frac{\partial I}{\partial y}
$$

- `Gx` measures change while moving left to right. It responds strongly to
  **vertical** boundaries.
- `Gy` measures change while moving top to bottom. It responds strongly to
  **horizontal** boundaries.
- The sign tells the direction of the brightness transition.

The gradient vector at a pixel is

$$
\nabla I = [G_x, G_y]^T.
$$



In [ ]:
# A simple 1D step edge and its forward difference.
signal = np.array([20, 20, 20, 20, 220, 220, 220, 220], dtype=float)
forward_difference = np.diff(signal, prepend=signal[0])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].stem(np.arange(signal.size), signal, basefmt=" ")
axes[0].set_title("1D intensity signal")
axes[0].set_xlabel("Pixel position x")
axes[0].set_ylabel("Intensity I(x)")
axes[0].set_ylim(-10, 250)

axes[1].stem(np.arange(signal.size), forward_difference, basefmt=" ")
axes[1].set_title("First difference: large at the edge")
axes[1].set_xlabel("Pixel position x")
axes[1].set_ylabel("I(x) - I(x-1)")
axes[1].axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Signal:            ", signal.astype(int))
print("Forward difference:", forward_difference.astype(int))



### Check your understanding

1. At which sample is the step edge detected?
2. What would happen to the sign if the signal changed from bright to dark?
3. Why does a constant region have derivative zero?



## 4. Correlation helper and padding

A derivative operator is a small kernel moved across the image. At each
location, the output is the sum of element-by-element products:

$$
R(r,c)=\sum_i\sum_j I(r+i,c+j)K(i,j).
$$

This notebook uses cross-correlation, so the kernel is not flipped. Flipping a
derivative kernel changes the sign of its response but not its edge magnitude.
Reflect padding reduces artificial borders compared with zero padding.



In [ ]:
def correlate2d(image, kernel, padding="reflect"):
    """Vectorized educational 2D cross-correlation with same-size output."""
    image = np.asarray(image, dtype=np.float32)
    kernel = np.asarray(kernel, dtype=np.float32)
    kh, kw = kernel.shape

    top = kh // 2
    bottom = kh - 1 - top
    left = kw // 2
    right = kw - 1 - left

    pad_mode = {"reflect": "reflect", "zero": "constant", "replicate": "edge"}
    if padding not in pad_mode:
        raise ValueError("padding must be 'reflect', 'zero', or 'replicate'")

    padded = np.pad(
        image,
        ((top, bottom), (left, right)),
        mode=pad_mode[padding],
    )
    windows = sliding_window_view(padded, (kh, kw))
    return np.einsum("rcij,ij->rc", windows, kernel, optimize=True)


def normalize_01(array, percentile=99.0):
    """Scale nonnegative values to [0, 1] using a robust upper limit."""
    array = np.asarray(array, dtype=np.float32)
    upper = float(np.percentile(array, percentile))
    if upper <= 1e-12:
        return np.zeros_like(array)
    return np.clip(array / upper, 0, 1)


def gradient_result(image, kx, ky, padding="reflect"):
    gx = correlate2d(image, kx, padding)
    gy = correlate2d(image, ky, padding)
    magnitude = np.hypot(gx, gy)
    direction = np.arctan2(gy, gx)
    return gx, gy, magnitude, direction


def show_signed(ax, response, title):
    limit = float(np.percentile(np.abs(response), 99))
    limit = max(limit, 1e-6)
    shown = ax.imshow(response, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(title)
    ax.axis("off")
    return shown


def show_gradient(name, image, kx, ky, padding="reflect"):
    gx, gy, magnitude, direction = gradient_result(image, kx, ky, padding)
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
    axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Input")
    axes[0].axis("off")
    show_signed(axes[1], gx, f"{name} Gx (signed)")
    show_signed(axes[2], gy, f"{name} Gy (signed)")
    axes[3].imshow(normalize_01(magnitude), cmap="magma", vmin=0, vmax=1)
    axes[3].set_title("Magnitude sqrt(Gx^2 + Gy^2)")
    axes[3].axis("off")
    plt.tight_layout()
    plt.show()
    return {"gx": gx, "gy": gy, "magnitude": magnitude, "direction": direction}



## 5. A hand-sized Sobel example

Consider a dark-to-bright vertical step. The Sobel `Gx` kernel gives extra
weight to the center row while differentiating left versus right.



In [ ]:
small_step = np.array([
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
    [10, 10, 10, 200, 200, 200, 200],
], dtype=np.float32)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1],
], dtype=np.float32)

small_gx = correlate2d(small_step, sobel_x, padding="replicate")
small_gy = correlate2d(small_step, sobel_y, padding="replicate")

print("Input image:")
print(small_step.astype(int))
print("\nSobel Gx response:")
print(small_gx.astype(int))
print("\nSobel Gy response:")
print(small_gy.astype(int))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title, cmap in zip(
    axes,
    [small_step, small_gx, small_gy],
    ["7 x 7 vertical step", "Sobel Gx", "Sobel Gy"],
    ["gray", "RdBu_r", "RdBu_r"],
):
    ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(7))
    ax.set_yticks(range(7))
    for r in range(7):
        for c in range(7):
            ax.text(c, r, f"{data[r, c]:.0f}", ha="center", va="center", fontsize=7)
plt.tight_layout()
plt.show()



**Interpretation:** `Gx` is large near the vertical transition because the
right side of the neighborhood is brighter than the left side. `Gy` is zero
because intensity does not change vertically.



## 6. First-derivative edge operators

All four operators below estimate `Gx` and `Gy`, but their kernel size and
weighting differ.

| Operator | Size | Main idea |
|---|---:|---|
| Roberts | 2 x 2 | Fast diagonal differences; sensitive to noise |
| Prewitt | 3 x 3 | Difference plus uniform smoothing |
| Sobel | 3 x 3 | Difference plus stronger center weighting |
| Scharr | 3 x 3 | Better rotational symmetry than 3 x 3 Sobel |



In [ ]:
KERNELS = {
    "Roberts": (
        np.array([[1, 0], [0, -1]], dtype=np.float32),
        np.array([[0, 1], [-1, 0]], dtype=np.float32),
    ),
    "Prewitt": (
        np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32),
        np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32),
    ),
    "Sobel": (sobel_x, sobel_y),
    "Scharr": (
        np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], dtype=np.float32),
        np.array([[-3, -10, -3], [0, 0, 0], [3, 10, 3]], dtype=np.float32),
    ),
}

for name, (kx, ky) in KERNELS.items():
    print(f"\n{name} Kx:\n{kx}")
    print(f"{name} Ky:\n{ky}")



### 6.1 Sobel: inspect the full gradient



In [ ]:
sobel = show_gradient("Sobel", gray, *KERNELS["Sobel"])



### 6.2 Gradient magnitude and direction

The gradient magnitude combines both components:

$$
M = \sqrt{G_x^2 + G_y^2}.
$$

The gradient direction is

$$
\theta = \operatorname{atan2}(G_y,G_x).
$$

The gradient points toward the greatest increase in brightness. The visible
edge itself is approximately perpendicular to that gradient direction.



In [ ]:
def direction_rgb(direction, magnitude):
    """Encode direction as hue and magnitude as brightness."""
    hue = ((direction + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    saturation = np.full_like(hue, 255, dtype=np.uint8)
    value = (normalize_01(magnitude) * 255).astype(np.uint8)
    hsv = np.dstack([hue, saturation, value])
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


direction_view = direction_rgb(sobel["direction"], sobel["magnitude"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(normalize_01(sobel["magnitude"]), cmap="magma", vmin=0, vmax=1)
axes[0].set_title("Sobel gradient magnitude")
axes[1].imshow(direction_view)
axes[1].set_title("Direction = hue, magnitude = brightness")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()



### 6.3 Compare Roberts, Prewitt, Sobel, and Scharr

Each magnitude image is normalized independently for visualization. Therefore,
compare edge continuity, thickness, noise response, and localization rather
than comparing display brightness alone.



In [ ]:
gradient_outputs = {}
for name, (kx, ky) in KERNELS.items():
    gx, gy, magnitude, direction = gradient_result(gray, kx, ky)
    gradient_outputs[name] = {
        "gx": gx,
        "gy": gy,
        "magnitude": magnitude,
        "direction": direction,
    }

fig, axes = plt.subplots(1, 5, figsize=(19, 4.3))
axes[0].imshow(gray_u8, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")
axes[0].axis("off")
for ax, (name, result) in zip(axes[1:], gradient_outputs.items()):
    ax.imshow(normalize_01(result["magnitude"]), cmap="magma", vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "first_derivative_comparison.png", dpi=160, bbox_inches="tight")
plt.show()



**Discuss:**

1. Which operator produces the cleanest continuous boundaries?
2. Which operator responds most strongly to fine texture or noise?
3. Do all operators place the main object boundaries at similar locations?
4. Why is comparing raw response values unfair when kernel scales differ?



## 7. From a response to a binary edge mask

A gradient image is not yet a binary edge map. We classify a pixel as an edge
when its magnitude exceeds a threshold:

$$
E(r,c)=\begin{cases}
1,&M(r,c)\ge T\\
0,&M(r,c)<T.
\end{cases}
$$

The slider threshold below is expressed as a percentage of the robustly
normalized magnitude, not as a universal physical value.



In [ ]:
sobel_strength = normalize_01(sobel["magnitude"])


def threshold_explorer(threshold_percent=25):
    threshold = threshold_percent / 100.0
    edge_mask = sobel_strength >= threshold
    density = 100 * edge_mask.mean()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(image_rgb)
    axes[0].set_title("Input")
    axes[1].imshow(sobel_strength, cmap="magma", vmin=0, vmax=1)
    axes[1].set_title("Normalized Sobel magnitude")
    axes[2].imshow(edge_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Mask: T={threshold_percent}% | edge pixels={density:.1f}%")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return edge_mask


try:
    from ipywidgets import interact, IntSlider
    interact(
        threshold_explorer,
        threshold_percent=IntSlider(
            value=25, min=1, max=90, step=1, description="Threshold"
        ),
    )
except ImportError:
    edge_mask = threshold_explorer(25)



**Observe:** A low threshold finds weak edges but also texture and noise. A high
threshold keeps only strong transitions but may break real boundaries.



## 8. Noise and smoothing before differentiation

Differentiation amplifies high-frequency changes, and random noise contains many
such changes. A Gaussian filter suppresses noise before the derivative is
computed.

A 2D Gaussian has the form

$$
G(x,y)=\frac{1}{2\pi\sigma^2}
\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right).
$$

Larger `sigma` removes more noise, but it also blurs and shifts fine details.



In [ ]:
def gaussian_kernel(size=5, sigma=1.2):
    if size % 2 == 0 or size < 3:
        raise ValueError("Gaussian kernel size must be an odd integer >= 3")
    radius = size // 2
    coordinates = np.arange(-radius, radius + 1, dtype=np.float32)
    xx, yy = np.meshgrid(coordinates, coordinates)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.sum()


rng = np.random.default_rng(441)
noise_sigma = 22.0
noisy = np.clip(gray + rng.normal(0, noise_sigma, gray.shape), 0, 255).astype(np.float32)
gaussian_5 = gaussian_kernel(5, sigma=1.2)
smoothed = correlate2d(noisy, gaussian_5, padding="reflect")

_, _, noisy_mag, _ = gradient_result(noisy, *KERNELS["Sobel"])
_, _, smooth_mag, _ = gradient_result(smoothed, *KERNELS["Sobel"])

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].imshow(noisy, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title(f"Noisy image (sigma={noise_sigma:.0f})")
axes[0, 1].imshow(smoothed, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Gaussian-smoothed noisy image")
axes[1, 0].imshow(normalize_01(noisy_mag), cmap="magma")
axes[1, 0].set_title("Sobel after no smoothing")
axes[1, 1].imshow(normalize_01(smooth_mag), cmap="magma")
axes[1, 1].set_title("Sobel after Gaussian smoothing")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

print("5 x 5 Gaussian kernel (sum should be 1):")
print(np.round(gaussian_5, 4))
print("Kernel sum:", gaussian_5.sum())



### Smoothing experiment

Change `sigma` and kernel size below. Keep the kernel odd and large enough to
contain the Gaussian (a common rule is size about `6*sigma + 1`, rounded up to
an odd number).



In [ ]:
def smoothing_explorer(sigma=1.2, kernel_size=5, edge_threshold=25):
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = gaussian_kernel(kernel_size, sigma)
    blurred = correlate2d(gray, kernel)
    _, _, magnitude, _ = gradient_result(blurred, *KERNELS["Sobel"])
    magnitude_01 = normalize_01(magnitude)
    mask = magnitude_01 >= edge_threshold / 100

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(blurred, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title(f"Gaussian: size={kernel_size}, sigma={sigma:.1f}")
    axes[1].imshow(magnitude_01, cmap="magma", vmin=0, vmax=1)
    axes[1].set_title("Sobel magnitude")
    axes[2].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Threshold={edge_threshold}%")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


try:
    from ipywidgets import FloatSlider, IntSlider, interact
    interact(
        smoothing_explorer,
        sigma=FloatSlider(value=1.2, min=0.5, max=4.0, step=0.1),
        kernel_size=IntSlider(value=5, min=3, max=15, step=2),
        edge_threshold=IntSlider(value=25, min=5, max=70, step=1),
    )
except ImportError:
    smoothing_explorer()



## 9. Second derivatives: Laplacian

The Laplacian adds second derivatives in `x` and `y`:

$$
\nabla^2 I = \frac{\partial^2 I}{\partial x^2}
+ \frac{\partial^2 I}{\partial y^2}.
$$

Unlike `Gx` and `Gy`, the Laplacian has no preferred direction. Around a step
edge, its signed response changes from positive to negative. The edge is often
located at the **zero crossing**, not simply at the largest absolute value.



In [ ]:
laplacian_4 = np.array([
    [0, 1, 0],
    [1, -4, 1],
    [0, 1, 0],
], dtype=np.float32)

laplacian_8 = np.array([
    [1, 1, 1],
    [1, -8, 1],
    [1, 1, 1],
], dtype=np.float32)

lap4_response = correlate2d(gray, laplacian_4)
lap8_response = correlate2d(gray, laplacian_8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(gray_u8, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")
show_signed(axes[1], lap4_response, "4-neighbor Laplacian")
show_signed(axes[2], lap8_response, "8-neighbor Laplacian")
axes[0].axis("off")
plt.tight_layout()
plt.show()



## 10. LoG and DoG zero-crossing detectors

### Laplacian of Gaussian (LoG)

1. Smooth with a Gaussian.
2. Apply the Laplacian.
3. Find locations where the response changes sign.

### Difference of Gaussians (DoG)

Subtract two images smoothed at different scales:

$$
DoG = G_{\sigma_1}*I - G_{\sigma_2}*I,
\qquad \sigma_2 > \sigma_1.
$$

DoG is an efficient approximation related to scale-normalized LoG. Both methods
need a contrast condition because tiny sign changes can be caused by noise.



In [ ]:
def zero_crossing(response, contrast_threshold=8.0):
    """Detect 3 x 3 sign changes with sufficient local response range."""
    padded = np.pad(response, 1, mode="reflect")
    windows = sliding_window_view(padded, (3, 3))
    local_min = windows.min(axis=(-2, -1))
    local_max = windows.max(axis=(-2, -1))
    sign_change = (local_min < 0) & (local_max > 0)
    enough_contrast = (local_max - local_min) >= contrast_threshold
    return sign_change & enough_contrast


log_blurred = correlate2d(gray, gaussian_kernel(7, sigma=1.4))
log_response = correlate2d(log_blurred, laplacian_8)
log_edges = zero_crossing(log_response, contrast_threshold=10)

g_sigma1 = correlate2d(gray, gaussian_kernel(7, sigma=1.0))
g_sigma2 = correlate2d(gray, gaussian_kernel(13, sigma=2.0))
dog_response = g_sigma1 - g_sigma2
dog_edges = zero_crossing(dog_response, contrast_threshold=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
show_signed(axes[0, 0], log_response, "LoG signed response")
axes[0, 1].imshow(log_edges, cmap="gray", vmin=0, vmax=1)
axes[0, 1].set_title("LoG zero crossings")
show_signed(axes[1, 0], dog_response, "DoG signed response")
axes[1, 1].imshow(dog_edges, cmap="gray", vmin=0, vmax=1)
axes[1, 1].set_title("DoG zero crossings")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()



**Think:** Why can thresholding `abs(Laplacian)` produce two responses around
one step boundary, while a zero crossing estimates the center of the change?



## 11. Canny edge detector

Canny is a multi-stage detector designed to produce thin, connected edges:

1. **Gaussian smoothing** reduces noise.
2. **Gradient calculation** estimates magnitude and direction.
3. **Non-maximum suppression** keeps only local maxima across the gradient
   direction, making edges thin.
4. **Double thresholding** labels strong, weak, and rejected pixels.
5. **Hysteresis** keeps weak pixels only when connected to strong edges.

`low_threshold` and `high_threshold` operate on OpenCV's internal gradient
scale. A useful starting ratio is `low:high` between `1:2` and `1:3`, but the
best values depend on image contrast and noise.



In [ ]:
def canny_explorer(low_threshold=50, high_threshold=150, blur_sigma=1.2):
    if low_threshold >= high_threshold:
        print("Choose low_threshold < high_threshold.")
        return

    blurred = cv2.GaussianBlur(gray_u8, (5, 5), blur_sigma)
    edges = cv2.Canny(
        blurred,
        threshold1=low_threshold,
        threshold2=high_threshold,
        apertureSize=3,
        L2gradient=True,
    )
    density = 100 * np.mean(edges > 0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(gray_u8, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Input")
    axes[1].imshow(blurred, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"Gaussian smoothing: sigma={blur_sigma:.1f}")
    axes[2].imshow(edges, cmap="gray", vmin=0, vmax=255)
    axes[2].set_title(
        f"Canny: low={low_threshold}, high={high_threshold}\n"
        f"edge pixels={density:.1f}%"
    )
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return edges


try:
    from ipywidgets import FloatSlider, IntSlider, interact
    interact(
        canny_explorer,
        low_threshold=IntSlider(value=50, min=0, max=250, step=5),
        high_threshold=IntSlider(value=150, min=5, max=300, step=5),
        blur_sigma=FloatSlider(value=1.2, min=0.1, max=4.0, step=0.1),
    )
except ImportError:
    canny_explorer()



### Why Canny edges are often thinner

A plain threshold on gradient magnitude can mark several neighboring pixels.
Canny's non-maximum suppression compares a pixel with neighbors along the
gradient direction and keeps it only if it is a local maximum. Hysteresis then
joins supported weak segments without accepting every weak response.



## 12. Final detector comparison

The following display uses one reasonable parameter setting. It is not proof
that one detector is always best. A good detector depends on the task:
measurement, segmentation, line detection, corner detection, or recognition.



In [ ]:
sobel_mask = normalize_01(gradient_outputs["Sobel"]["magnitude"]) >= 0.25
scharr_mask = normalize_01(gradient_outputs["Scharr"]["magnitude"]) >= 0.25
canny_edges = cv2.Canny(
    cv2.GaussianBlur(gray_u8, (5, 5), 1.2),
    50,
    150,
    apertureSize=3,
    L2gradient=True,
)

comparison = [
    ("Input", image_rgb, None),
    ("Sobel magnitude", normalize_01(gradient_outputs["Sobel"]["magnitude"]), "magma"),
    ("Sobel threshold", sobel_mask, "gray"),
    ("Scharr threshold", scharr_mask, "gray"),
    ("LoG zero crossings", log_edges, "gray"),
    ("DoG zero crossings", dog_edges, "gray"),
    ("Canny", canny_edges, "gray"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (title, data, cmap) in zip(axes.ravel(), comparison):
    ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "edge_detector_comparison.png", dpi=170, bbox_inches="tight")
plt.show()

for title, data, _ in comparison[2:]:
    density = 100 * np.mean(np.asarray(data) > 0)
    print(f"{title:22s}: {density:5.2f}% edge pixels")



## 13. Use an edge mask

An edge map can be used to highlight boundaries on the original image. Here,
Canny edges are drawn in red. This is a visualization operation; it does not
change how the edges were detected.



In [ ]:
edge_boolean = canny_edges > 0
overlay = image_rgb.copy()
overlay[edge_boolean] = np.array([255, 35, 35], dtype=np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(canny_edges, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Binary Canny mask")
axes[1].imshow(overlay)
axes[1].set_title("Detected edges overlaid in red")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "canny_overlay.png", dpi=170, bbox_inches="tight")
plt.show()

cv2.imwrite(str(OUTPUT_DIR / "canny_edges.png"), canny_edges)
cv2.imwrite(
    str(OUTPUT_DIR / "edge_overlay.png"),
    cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR),
)
print("Saved outputs in:", OUTPUT_DIR.resolve())



## 14. Common interpretation mistakes

1. **"A bright response is a bright object."** No. A bright magnitude means a
   strong local change, not high original intensity.
2. **"Gx finds horizontal edges."** `Gx` differentiates horizontally and is
   strongest at vertical boundaries.
3. **"Gradient direction runs along the edge."** It normally points across the
   edge, perpendicular to the boundary.
4. **"The Laplacian magnitude alone locates the edge center."** A zero crossing
   is usually the relevant location for a second-derivative detector.
5. **"More detected pixels means a better detector."** Extra pixels may be
   noise, texture, double responses, or thick boundaries.
6. **"One threshold works for every image."** Thresholds depend on contrast,
   noise, exposure, preprocessing, and the final task.



## 15. Student investigation

Upload two different images and record your answers.

### Task A: operator comparison

1. Identify one vertical and one horizontal boundary in the image.
2. Compare their responses in `Gx` and `Gy`.
3. Rank Roberts, Prewitt, Sobel, and Scharr for noise sensitivity.
4. Explain one difference using the kernel weights.

### Task B: smoothing and scale

1. Run Sobel with Gaussian `sigma = 0.5, 1.5, 3.0`.
2. Keep the threshold fixed.
3. Record edge-pixel percentage and describe lost or improved boundaries.
4. Explain the tradeoff between noise removal and localization.

### Task C: Canny thresholds

1. Keep `high=150`; test `low=20, 50, 100`.
2. Keep `low=50`; test `high=80, 150, 240`.
3. Identify weak edges preserved through hysteresis.
4. Choose final settings and justify them from the intended application.

### Task D: first versus second derivatives

1. Compare Sobel magnitude, LoG zero crossings, and Canny.
2. Find an edge where the methods disagree.
3. Explain whether the disagreement is caused by smoothing, derivative order,
   thresholding, non-maximum suppression, or hysteresis.



## 16. Short calculation problems

**Problem 1.** For `Gx = 30` and `Gy = 40`, calculate gradient magnitude and
direction in degrees.

**Problem 2.** A pixel has `M = 72`. Classify it as rejected, weak, or strong
for Canny thresholds `T_low = 50` and `T_high = 100`.

**Problem 3.** Explain why the kernel sum of Sobel `Kx` is zero.

**Problem 4.** Predict the sign of `Gx` for a transition from bright on the left
to dark on the right using the Sobel kernel shown in this notebook.

**Problem 5.** A LoG neighborhood contains values `[-3, -1, 2, 4]`. Does it
contain a sign change? What additional condition should be checked before
accepting an edge?



### Solutions

**1.**

$$M=\sqrt{30^2+40^2}=50,$$

$$\theta=\operatorname{atan2}(40,30)=53.13^\circ.$$

**2.** It is a **weak** candidate because `50 <= 72 < 100`. Hysteresis keeps it
only if it connects to a strong edge.

**3.** A constant neighborhood should produce zero derivative. The negative and
positive weights cancel, so the kernel sum is zero.

**4.** The response is negative because the positively weighted right side is
darker than the negatively weighted left side. Reversing the kernel convention
would reverse the sign but preserve magnitude.

**5.** Yes, the values cross from negative to positive. Also require enough
local contrast (for example, `max-min` above a threshold) to reject tiny noisy
sign changes.



## 17. Summary

- Edges are rapid spatial intensity changes.
- `Gx` and `Gy` are signed directional derivatives.
- Magnitude measures edge strength; direction describes the direction of
  greatest intensity increase.
- Roberts, Prewitt, Sobel, and Scharr use different derivative approximations.
- Smoothing helps because derivatives amplify noise.
- Laplacian, LoG, and DoG use second-derivative behavior and zero crossings.
- Canny adds smoothing, non-maximum suppression, double thresholds, and
  hysteresis to create thin, connected edges.
- The best parameters depend on the image and the downstream vision task.
